
# A morphological atlas: E, Sa, Sb, Sc, Im galaxy SEDs

The Coleman, Wu & Weedman 1980 spectral templates remain the textbook
illustration of how the integrated SED morphs along the Hubble sequence
— from quiescent ellipticals with deep 4000 Å breaks to gas-rich
irregulars dominated by ongoing star formation and nebular emission.

We synthesize the five canonical types within tengri using a single
parametric SFH (truncated skew-normal) and tune three knobs per type:
the SFH peak epoch, the SFH width, and the dust optical depth. The
emerging atlas shows the same chromatic ordering as Coleman+1980:
ellipticals are reddest at all wavelengths, irregulars dominate the UV.

References:

- Coleman, Wu & Weedman 1980, ApJS, 43, 393
- Kennicutt 1992, ApJS, 79, 255 (modern revision of the atlas)


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# (label, peak_lbt_gyr, width_gyr, tau_diff, color)
ATLAS = [
    ("Elliptical", 9.0, 1.5, 0.05, "#cc3333"),
    ("Sa (early spiral)", 7.0, 2.0, 0.20, "#ee8833"),
    ("Sb (spiral)", 5.0, 2.5, 0.35, "#aabb33"),
    ("Sc (late spiral)", 3.0, 2.5, 0.50, "#3399aa"),
    ("Im (irregular)", 1.0, 2.0, 0.10, "#5566cc"),
]

ssp = tengri.load_ssp()
model = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "tsnorm",
        "all_params": tengri.FIXED,
        "peak_lbt_gyr": tengri.Uniform(0.1, 13.0),
        "width_gyr": tengri.Uniform(0.1, 5.0),
        "log_total_mass": 10.0,
        "skew": 0.0,
        "trunc": 13.5,
    },
    dust={
        "type": "two_component",
        "all_params": tengri.FIXED,
        "tau_diff": tengri.Uniform(0.0, 2.0),
        "tau_bc": 0.3,
        "slope": -0.7,
    },
    redshift=tengri.Fixed(0.0),
)
baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))

C_AA_PER_S = 2.998e18
fig, ax = plt.subplots(figsize=(7.2, 4.8))
for label, peak, width, tau, color in ATLAS:
    params = {
        **baseline,
        "sfh_tsnorm_peak_lbt_gyr": jnp.float64(peak),
        "sfh_tsnorm_width_gyr": jnp.float64(width),
        "dust_tau_diff": jnp.float64(tau),
    }
    out = model.predict(params)
    wave = np.asarray(model.wavelengths)
    nu = C_AA_PER_S / wave
    nu_l_nu = nu * np.asarray(out.rest_sed())
    # Normalize each spectrum to its 5500 Å value so the chromatic
    # ordering — not the absolute luminosity — reads cleanly.
    norm = nu_l_nu[np.argmin(np.abs(wave - 5500.0))]
    ax.loglog(wave, nu_l_nu / norm, color=color, lw=1.6, label=label)

ax.axvline(4000, color="0.5", lw=0.6, ls=":")
ax.text(4000, 0.025, "4000 Å break", color="0.4", fontsize=8, rotation=90, va="bottom", ha="right")
ax.axvline(1216, color="0.5", lw=0.6, ls=":")
ax.text(1216, 0.025, r"Ly$\alpha$", color="0.4", fontsize=8, rotation=90, va="bottom", ha="right")
ax.set_xlim(900, 5e4)
ax.set_ylim(2e-2, 30.0)
ax.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]")
ax.set_ylabel(r"$\nu L_\nu\,/\,\nu L_\nu\,(5500\,\mathrm{\AA})$")
ax.legend(frameon=False, fontsize=9, loc="upper right")

fig.tight_layout()
plt.savefig("plot_usecase_hubble_sequence.png", dpi=150, bbox_inches="tight")